---
title: Aproximando las horas de luz diurna
subject: Aprendizaje Profundo
subtitle: 
short_title: Ejemplo aproximador universal
authors:
  - name: Jorge Anais
    orcid: 0000-0001-9051-1338
    email: jrganais@gmail.com
license: MIT
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jorgeanais/libro_aprendizaje_profundo/blob/main/cap1/160_horas_luz_chile.ipynb)


**Objetivo:** demostrar de que una red neuronal *feed-forward* (perceptrón multicapa) es capaz de aprender a aproximar una función a partir de un conjunto de datos de ejemplo.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from bokeh.plotting import figure, show, output_notebook
from bokeh.layouts import gridplot
from bokeh.palettes import Category10

from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

output_notebook()
np.random.seed(42)


Loading BokehJS ...

## 1. Luz diurna

En este cuadernillo utilizaremos de ejemplo la duración de la luz diurna, según la latitud y momento del año. Usamos la aproximación astronómica estándar para el largo del día:

1. **Declinación solar** $\delta$ (el ángulo entre los rayos del sol y el plano ecuatorial), que varía a lo largo del año siguiendo aproximadamente una sinusoide (aproximación de Cooper, 1969):

$$\delta(N) = 23.45° \cdot \sin\left(\frac{360}{365}(284 + N)\right)$$

donde $N$ es el día del año (1 a 365).

2. El **ángulo horario de salida/puesta del sol** $\omega_0$, que depende de la latitud $\varphi$ y de la declinación:

$$\cos(\omega_0) = -\tan(\varphi)\cdot\tan(\delta)$$

3. Finalmente, las **horas de luz** son:

$$\text{Horas de luz} = \frac{24}{\pi}\,\omega_0 \quad (\omega_0 \text{ en radianes})$$

Esta es una función bastante "no trivial": combina senos, tangentes y arcocosenos, y no es
para nada una relación lineal ni polinomial simple entre el día del año, la latitud y las horas de luz. Por este motivo es que lo usaremos de ejemplo.

In [2]:
def horas_luz(dia_del_anio, latitud_grados):
    '''
    Calcula las horas de luz diurna dado el día del año (1-365) y la latitud en grados
    (negativa para el hemisferio sur), usando la aproximación astronómica estándar.
    '''
    lat_rad = np.radians(latitud_grados)

    # Declinación solar (aproximación de Cooper, 1969)
    declinacion_grados = 23.45 * np.sin(np.radians(360 / 365 * (284 + dia_del_anio)))
    declinacion_rad = np.radians(declinacion_grados)

    # Ángulo horario de salida del sol
    cos_omega = -np.tan(lat_rad) * np.tan(declinacion_rad)
    cos_omega = np.clip(cos_omega, -1, 1)  # evita errores numéricos en latitudes extremas
    omega = np.arccos(cos_omega)

    return (24 / np.pi) * omega


In [3]:
ciudades = {
    "Arica": -18.48,
    "Santiago": -33.45,
    "Punta Arenas": -53.16,
}

dias = np.arange(1, 366)
colores = Category10[3]

p1 = figure(
    title="Horas de luz diurna a lo largo del año (modelo analítico)",
    x_axis_label="Día del año", y_axis_label="Horas de luz",
    width=800, height=450,
)

for (ciudad, lat), color in zip(ciudades.items(), colores):
    p1.line(dias, horas_luz(dias, lat), line_width=2.5, color=color,
             legend_label=f"{ciudad} (lat {lat}°)")

p1.legend.location = "bottom_right"
p1.legend.click_policy = "hide"
show(p1)


Se nota claramente que, mientras más cerca del ecuador (Arica), la curva es casi plana
(≈12 horas todo el año), y mientras más al sur (Punta Arenas), la variación estacional es enorme
(de ~7 a ~17 horas).

## 2. Simular datos

Consideremos que alguien anota la hora de amanecer y atardecer algunos días del año, con cierto error de medición. Vamos a simular esto, tomamos algunos días al azar por ciudad y le agregamos un pequeño ruido (gaussiano).

In [4]:
def generar_datos(ciudades, n_muestras_por_ciudad=40, ruido_std=0.15):
    filas = []
    for ciudad, lat in ciudades.items():
        dias_muestreados = np.random.choice(np.arange(1, 366), size=n_muestras_por_ciudad, replace=False)
        horas_verdaderas = horas_luz(dias_muestreados, lat)
        horas_observadas = horas_verdaderas + np.random.normal(0, ruido_std, size=n_muestras_por_ciudad)
        for d, h in zip(dias_muestreados, horas_observadas):
            filas.append({"dia": d, "latitud": lat, "ciudad": ciudad, "horas_luz": h})
    return pd.DataFrame(filas)

df = generar_datos(ciudades, n_muestras_por_ciudad=40, ruido_std=0.15)
df.head(10)


,dia,latitud,ciudad,horas_luz
0,194,-18.48,Arica,10.796926
1,34,-18.48,Arica,12.854696
2,16,-18.48,Arica,12.962947
3,310,-18.48,Arica,12.670524
4,58,-18.48,Arica,12.416826
5,184,-18.48,Arica,10.852661
6,77,-18.48,Arica,12.355348
7,120,-18.48,Arica,11.233360
8,153,-18.48,Arica,10.819159
9,127,-18.48,Arica,11.165041


In [5]:
p2 = figure(
    title="Datos de ejemplo vs. curva real",
    x_axis_label="Día del año", y_axis_label="Horas de luz",
    width=800, height=450,
)

for (ciudad, lat), color in zip(ciudades.items(), colores):
    p2.line(dias, horas_luz(dias, lat), line_width=1.5, color=color, alpha=0.35,
             legend_label=f"{ciudad} (real)")
    sub = df[df.ciudad == ciudad]
    p2.scatter(sub.dia, sub.horas_luz, size=6, color=color,
                legend_label=f"{ciudad} (datos)")

p2.legend.location = "bottom_right"
p2.legend.click_policy = "hide"
show(p2)


## 3. Entrenar una red neuronal feed-forward simple

Usaremos `MLPRegressor` de scikit-learn: un perceptrón multicapa (feed-forward) sencillo, con un par de capas ocultas.

In [6]:
X = df[["dia", "latitud"]].values
y = df["horas_luz"].values

# Es buena práctica normalizar las entradas antes de entrenar una red neuronal
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

modelo = MLPRegressor(
    hidden_layer_sizes=(32, 32),   # dos capas ocultas de 32 neuronas: red pequeña
    activation="tanh",
    solver="lbfgs",                # buen solver para datasets chicos como este
    max_iter=5000,
    random_state=42,
)
modelo.fit(X_scaled, y)

y_pred_train = modelo.predict(X_scaled)
print(f"R² en datos de entrenamiento:  {r2_score(y, y_pred_train):.4f}")
print(f"MAE en datos de entrenamiento: {mean_absolute_error(y, y_pred_train):.4f} horas")


R² en datos de entrenamiento:  0.9983
MAE en datos de entrenamiento: 0.0633 horas


## 4. Comparación: función real vs. datos vs. red neuronal

Para "visualizar lo aprendido", le pedimos a la red que prediga las horas de luz para
**todos** los días del año (1 a 365) en cada ciudad, y comparamos esa curva con la curva
analítica real y con los datos de entrenamiento.

In [10]:
graficos = []
metricas = []

for (ciudad, lat), color in zip(ciudades.items(), colores):
    y_real = horas_luz(dias, lat)

    X_grid = np.column_stack([dias, np.full_like(dias, lat, dtype=float)])
    X_grid_scaled = scaler_X.transform(X_grid)
    y_nn = modelo.predict(X_grid_scaled)

    sub = df[df.ciudad == ciudad]

    p = figure(title=f"{ciudad} (lat {lat}°)", x_axis_label="Día del año",
               y_axis_label="Horas de luz", width=800, height=450)

    p.line(dias, y_nn, line_width=1.0, color=color, legend_label="Red neuronal")
    p.line(dias, y_real, line_width=1.0, color="grey", legend_label="Función real", line_dash="dashed")
    p.scatter(sub.dia, sub.horas_luz, size=6, color=color, alpha=0.6, legend_label="Datos de entrenamiento")
    p.legend.location = "bottom_right"
    p.legend.label_text_font_size = "7pt"
    graficos.append(p)

    metricas.append({
        "ciudad": ciudad,
        "MAE (horas)": mean_absolute_error(y_real, y_nn),
        "R2": r2_score(y_real, y_nn),
    })

grid = gridplot(graficos, ncols=1, sizing_mode="scale_width")
show(grid)

pd.DataFrame(metricas)


,ciudad,MAE (horas),R2
0,Arica,0.082322,0.984896
1,Santiago,0.259336,0.915469
2,Punta Arenas,0.248993,0.981805


## 5. Conclusión

la idea central de las redes neuronales feed-forward como
*aproximadores universales de funciones* es que dado un número suficiente de ejemplos, pueden aprender relaciones complejas entre entradas y salidas sin que nosotros les entreguemos la fórmula de manera explícita — algo muy útil cuando la relación real es desconocida o demasiado compleja de derivar
a mano (como ocurre en muchos problemas reales: precios, imágenes, texto, etc.).

### Para explorar
- ¿Qué pasa si reducimos el número de datos de entrenamiento (por ejemplo, a 5 por ciudad)?
- ¿Qué pasa si aumentamos el ruido?
- ¿Qué pasa si usamos una red más chica (por ejemplo, `hidden_layer_sizes=(4,)`) o más grande?
- ¿La red logra "extrapolar" bien a una latitud que nunca vio en el entrenamiento (por ejemplo, -70°)?
